## Setup do Ambiente
- Importação de bibliotecas essenciais do PySpark
- Definição dos paths utilizados para as tabelas
- Utilização do catálogo `catalogo`, schemas `silver_db_name`, `gold_db_name`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    explode,
    sequence,
    col,
    count,
    min,
    round,
    max,
    sum,
    struct,
    concat_ws,
    collect_list,
    year,
    quarter,
    month,
    weekofyear,
    dayofmonth,
    dayofweek,
    when,
    to_date,
    
)
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
from functools import reduce
from pyspark.sql.utils import AnalysisException

### Definição de Variáveis Globais
- Centralizamos os nomes de catálogos, bancos de dados e caminhos.
- **Boas Práticas:** Evitar "hardcoding" (escrever o caminho diretamente no código várias vezes). Se o nome do catálogo mudar no futuro, alteramos apenas aqui.

In [0]:
catalogo = "medalhao_credit"
silver_db_name = "silver_credit"
gold_db_name = "gold_credit"

In [0]:
spark.sql(f"USE CATALOG {catalogo};")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_db_name};")
spark.sql(f"USE SCHEMA {gold_db_name};")

## Funções Úteis

### Função `table_check`

Esta função verifica se uma tabela existe em um banco de dados específico e se ela contém dados.

In [0]:
def table_check(table_name, db_name):
    """
    Verifica se uma tabela existe e possui dados em um banco de dados especificado.

    Args:
        table_name (str): Nome da tabela a ser verificada.
        db_name (str): Nome do banco de dados onde a tabela está localizada.

    Returns:
        bool: True se a tabela existe e possui dados, False caso contrário.

    Raises:
        ValueError: Se a tabela existe mas está vazia.
    """
    if spark.catalog.tableExists(f"{db_name}.{table_name}"):
        if spark.table(f"{db_name}.{table_name}").count() == 0:
            raise ValueError(f"Tabela {db_name}.{table_name} existe mas está vazia.")
        return True
    return False

### Função `save_table_gold` 
Salva uma tabela a partir do catálogo (`catalogo`) e camada (`gold_db_name`) especificados, em formato Delta.
Adiciona a coluna `data_criacao_gold`

In [0]:
def save_table_gold(table_name: str, df, process_col=True):
    """
    Salva um DataFrame como tabela Delta na camada gold.

    Args:
        table_name (str): Nome da tabela a ser criada ou sobrescrita na camada gold.
        df (DataFrame): DataFrame Spark que será salvo como tabela Delta.
        process_col(Bool): Adicionar uma coluna com o current timestamp.
    """
    table_path = f"{catalogo}.{gold_db_name}.{table_name}"

    try:
        old_schema = spark.table(table_path).schema.simpleString()
    except AnalysisException:
        old_schema = None

    if process_col == True:
        df = df.withColumn("data_criacao_gold", current_timestamp())
    
    try:
        df.write \
            .format("delta") \
            .option("overwriteSchema", "true") \
            .mode("overwrite") \
            .saveAsTable(table_path)
    except Exception as ex:
        print(f"Erro ao salvar a tabela {table_path}: {ex}")
        return
    
    new_schema = spark.table(table_path).schema.simpleString()

    if old_schema is None:
        print(f"Tabela {table_path} criada pela primeira vez.")
    elif old_schema != new_schema:
        print(f"Esquema da tabela {table_path} foi alterado.\n")
        print("Schema anterior:")
        print(old_schema)
        print("\nNovo schema:")
        print(new_schema)
    else:
        print(f"Tabela salva com sucesso: {table_path}")

### Função `read_table`
Lê uma tabela do Databricks a partir do catálogo e camada especificados (`silver` ou `gold`), retornando um DataFrame Spark correspondente.

In [0]:
def read_table(nome_tabela: str, camada:str='silver'):
    """
    Lê uma tabela Delta da camada especificada.

    Args:
        nome_tabela (str): Nome da tabela a ser lida.
        camada (str, optional): Camada de origem da tabela ('silver' ou 'gold'). Default é 'bronze'.

    Returns:
        DataFrame: DataFrame Spark da tabela lida.

    Raises:
        ValueError: Se a camada não for 'silver' ou 'gold'.
        ValueError: Se a tabela não existir ou estiver vazia.
    """
    db_map = {
        'silver': silver_db_name,
        'gold': gold_db_name
    }

    db_nome = db_map.get(camada.lower())
    
    if not db_nome:
        raise ValueError("Camada deve ser 'silver' ou 'gold'")
    
    if not table_check(nome_tabela, db_nome):
        raise ValueError(f"Tabela {db_nome}.{nome_tabela} não existe.")
    return spark.table(f"{catalogo}.{db_nome}.{nome_tabela}")

##Views

In [0]:
df_jornada = read_table('ft_jornada_atendimento','gold')
df_tempo = read_table('dm_tempo','gold')
df_jornada.printSchema()
df_tempo.printSchema()

### View Operacional por Tipo de Atendimento
- **Objetivo**:  comparar jornadas que passaram por atendimento humano versus jornadas atendidas apenas por bot, divididas por status final de resolução.
- Passos: 
  -  Cria uma coluna categórica baseada na flag `flg_passou_humano`
  - Agrupa por tipo de atendimento e status de resolução
- Resultado:
  - `total_tipo:` Total de jornadas dentro de cada tipo de atendimento
  -` perc_status:` Percentual que cada status representa dentro do seu tipo
| Métrica | Descrição | Fórmula |
|---------|-----------|---------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) |
| `custo_total` | Custo acumulado | SUM(custo_total_jornada) |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) |
| `tentativas_max` | Máximo de tentativas | MAX(qtd_tentativas) |


In [0]:
df_operacional = df_jornada \
    .withColumn("tipo_atendimento", 
                F.when(F.col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .groupBy("tipo_atendimento", "status_final_resolucao") \
    .agg(
        F.count("*").alias("qtd_jornadas"),
        F.round(F.avg("custo_total_jornada"), 2).alias("custo_medio"),
        F.round(F.sum("custo_total_jornada"), 2).alias("custo_total"),
        F.round(F.avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        F.round(F.max("qtd_tentativas"), 2).alias("tentativas_media")
    ) \
    .withColumn("total_tipo", F.sum("qtd_jornadas").over(Window.partitionBy("tipo_atendimento"))) \
    .withColumn("perc_status", F.round((F.col("qtd_jornadas") / F.col("total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .withColumn("custo_total_tipo", F.round(F.sum("custo_total").over(Window.partitionBy("tipo_atendimento")), 2)) \
    .withColumn("perc_custo", F.round((F.col("custo_total") / F.col("custo_total_tipo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("tipo_atendimento", F.desc("qtd_jornadas"))

df_operacional.display()

### Análise Detalhada por Status de Resolução e Motivo

- **Objetivo:**
Analisar a distribuição de jornadas por **status de resolução** e **motivo de contato**, permitindo identificar quais motivos têm maior taxa de resolução, custo e necessidade de intervenção humana.
- **Passos:**
Agrupar por status de resolução e motivos.

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) | Volume absoluto dessa combinação |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) | Quanto custa em média essa jornada |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Quanto tempo demora para resolver |
| `tentativas_media` | Média de tentativas | AVG(qtd_tentativas) | Quantas vezes o cliente precisou tentar |
| `perc_passou_humano` | % com atendimento humano | (SUM(flg_passou_humano) / COUNT(*)) × 100 | Taxa de escalonamento para humano |
| `perc_dentro_status` | %  cada motivo representa dentro do seu status | Window function soma todas as jornadas do mesmo status | "Do total de jornadas resolvidas, 30% foram sobre Boleto"|
| `perc_total` | Calcula quanto cada combinação status+motivo representa do total geral|  Window.partitionBy(F.lit(1)) cria uma janela sobre toda a tabela | "Esta combinação representa 6% de todas as jornadas" |


In [0]:
df_status_motivo = df_jornada \
    .groupBy("status_final_resolucao", "motivo") \
    .agg(
        F.count("*").alias("qtd_jornadas"),
        F.round(F.avg("custo_total_jornada"), 2).alias("custo_medio"),
        F.round(F.avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        F.round(F.avg("qtd_tentativas"), 2).alias("tentativas_media"),
        F.round((F.sum("flg_passou_humano") / F.count("*") * 100).cast("decimal(5,2)"), 2).alias("perc_passou_humano")
    ) \
    .withColumn("perc_dentro_status", F.round((F.col("qtd_jornadas") /  F.sum("qtd_jornadas").over(Window.partitionBy("status_final_resolucao"))* 100).cast("decimal(5,2)"), 2)) \
    .withColumn("perc_total", 
    F.round((F.col("qtd_jornadas") / 
             F.sum("qtd_jornadas").over(Window.partitionBy(F.lit(1))) * 100)
            .cast("decimal(5,2)"), 2)) \
    .orderBy("status_final_resolucao", F.desc("qtd_jornadas"))

df_status_motivo.display()


### Análise de Satisfação do Cliente por Motivo

- **Objetivo:**
Analisar como os clientes avaliam sua experiência em cada **motivo de contato**, correlacionando satisfação com métricas operacionais como custo, duração e tipo de atendimento.

- **Passos:**
1. Filtrar apenas jornadas onde o cliente avaliou (`satisfacao_cliente IS NOT NULL`)
2. Agrupar por motivo e satisfação do cliente
3. Calcular métricas operacionais por combinação motivo + satisfação
4. Adicionar contexto percentual dentro de cada motivo

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_avaliacoes` | Total de avaliações | COUNT(*) | Volume de feedback nessa combinação |
| `custo_medio` | Custo médio | AVG(custo_total_jornada) | Correlacionar custo × satisfação |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Ver se tempo impacta satisfação |
| `perc_passou_humano` | % com atendimento humano | (SUM(flg_passou_humano) / COUNT(*)) × 100 | Verificar se humano melhora satisfação |
| `total_motivo` | Total de avaliações do motivo | Window function soma todas avaliações do mesmo motivo | Base para cálculo percentual |
| `perc_satisfacao` | % que cada nota representa no motivo | (qtd_avaliacoes / total_motivo) × 100 | "70% dos clientes que avaliaram Boleto ficaram satisfeitos" |


In [0]:
df_satisfacao_motivo = df_jornada \
    .filter(F.col("satisfacao_cliente").isNotNull()) \
    .groupBy("motivo", "satisfacao_cliente") \
    .agg(
        F.count("*").alias("qtd_avaliacoes"),
        F.round(F.avg("custo_total_jornada"), 2).alias("custo_medio"),
        F.round(F.avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        F.round((F.sum("flg_passou_humano") / F.count("*") * 100).cast("decimal(5,2)"), 2).alias("perc_passou_humano")
    ) \
    .withColumn("total_motivo", F.sum("qtd_avaliacoes").over(Window.partitionBy("motivo"))) \
    .withColumn("perc_satisfacao", F.round((F.col("qtd_avaliacoes") / F.col("total_motivo") * 100).cast("decimal(5,2)"), 2)) \
    .orderBy("motivo", "satisfacao_cliente")

df_satisfacao_motivo.display()


### Análise de Eficiência por Tipo de Atendimento

- **Objetivo:**
Comparar a **eficiência operacional** de jornadas atendidas apenas por bot versus jornadas que passaram por atendimento humano, segmentadas por motivo de contato.

- **Passos:**
1. Agrupar por motivo e flag de atendimento humano
2. Calcular métricas de performance (duração, custo, tentativas)
3. Calcular custo por minuto para medir eficiência
4. Classificar tipo de atendimento para melhor legibilidade
5. Ordenar por motivo e tipo para facilitar comparação

- **Resultados:**

| Métrica | Descrição | Fórmula | Interpretação |
|---------|-----------|---------|---------------|
| `qtd_jornadas` | Total de jornadas | COUNT(*) | Volume em cada tipo de atendimento |
| `duracao_media_min` | Tempo médio em minutos | AVG(duracao_jornada_minutos) | Quanto tempo demora para resolver |
| `custo_medio` | Custo médio por jornada | AVG(custo_total_jornada) | Quanto custa em média essa jornada |
| `tentativas_media` | Média de tentativas | AVG(qtd_tentativas) | Quantas vezes o cliente precisou tentar |
| `tipo_atendimento` | Classificação do atendimento | "Humano + Bot" ou "Apenas Bot" | Facilita leitura e análise |

In [0]:
df_eficiencia = df_jornada \
    .groupBy("motivo", "flg_passou_humano") \
    .agg(
        F.count("*").alias("qtd_jornadas"),
        F.round(F.avg("duracao_jornada_minutos"), 2).alias("duracao_media_min"),
        F.round(F.avg("custo_total_jornada"), 2).alias("custo_medio"),
        F.round(F.avg("qtd_tentativas"), 2).alias("tentativas_media")
    ) \
    .withColumn("tipo_atendimento", 
                F.when(F.col("flg_passou_humano") == 1, "Humano + Bot").otherwise("Apenas Bot")) \
    .select("motivo", "tipo_atendimento", "qtd_jornadas", "duracao_media_min", "custo_medio", "tentativas_media") \
    .orderBy("motivo", "tipo_atendimento")

df_eficiencia.display()

In [0]:
save_table_gold(df_operacional, "mview_operacional", False)
save_table_gold(df_status_motivo, "mview_status_motivo", False)
save_table_gold(df_satisfacao_motivo, "mview_satisfacao_motivo", False)
save_table_gold(df_eficiencia, "mview_eficiencia", False)